In [1]:
import os

BASE_PATH = "/kaggle/input/datasets/vikasn2006/shanghai-crowd"
os.chdir(BASE_PATH)

In [2]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from scipy.io import loadmat
from scipy.ndimage import gaussian_filter
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
print("hi")

hi


In [4]:
def generate_density_map(image, points):
    density = np.zeros((image.shape[0], image.shape[1]))

    for p in points:
        x = min(int(p[0]), image.shape[1] - 1)
        y = min(int(p[1]), image.shape[0] - 1)
        density[y, x] = 1

    density = gaussian_filter(density, sigma=3)
    return density

In [5]:
class CrowdDataset(Dataset):
    def __init__(self, img_folder, mat_folder=None):
        self.img_folder = img_folder
        self.mat_folder = mat_folder if mat_folder else img_folder
        self.img_paths = [img for img in os.listdir(img_folder) if img.endswith('.jpg')]

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_name = self.img_paths[idx]
        img_path = os.path.join(self.img_folder, img_name)

        mat_name = 'GT_' + img_name.replace('.jpg', '.mat')
        mat_path = os.path.join(self.mat_folder, mat_name)

        if not os.path.exists(mat_path):
            mat_name = img_name.replace('.jpg', '.mat')
            mat_path = os.path.join(self.mat_folder, mat_name)

        if not os.path.exists(mat_path):
            raise FileNotFoundError(f"Annotation file not found in: {mat_path}")

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]

        mat = loadmat(mat_path)
        points = mat["image_info"][0][0][0][0][0]

        # Generate density map at original scale
        density_map = generate_density_map(image, points)

        # CSRNet frontend (VGG16) downsamples by 8x.
        # We must resize GT to match model output: (H/8, W/8)
        # We multiply by 64 to preserve the sum (count) after downsampling
        target_h, target_w = h // 8, w // 8
        density_map = cv2.resize(density_map, (target_w, target_h), interpolation=cv2.INTER_CUBIC) * 64

        image = torch.tensor(image).permute(2,0,1).float() / 255.
        density_map = torch.tensor(density_map).unsqueeze(0).float()

        return image, density_map

In [6]:
import torch
import torch.nn as nn
from torchvision.models import vgg16

class CSRNet(nn.Module):
    def __init__(self, load_weights=True, weights_path=None, freeze_frontend=False):
        super(CSRNet, self).__init__()

        # ✅ Load VGG16 without internet dependency
        vgg = vgg16(weights=None)

        # ✅ Load pretrained weights manually (if provided)
        if load_weights and weights_path is not None:
            state_dict = torch.load(weights_path, map_location="cpu")
            vgg.load_state_dict(state_dict)
            print("✅ Pretrained VGG16 weights loaded")

        # ✅ Extract frontend (first 23 layers)
        self.frontend = vgg.features[:23]

        # ✅ Option to freeze frontend (transfer learning)
        if freeze_frontend:
            for param in self.frontend.parameters():
                param.requires_grad = False
            print("🔒 Frontend frozen")

        # ✅ CSRNet backend (dilated convs)
        self.backend = nn.Sequential(
            nn.Conv2d(512, 512, 3, padding=2, dilation=2),
            nn.ReLU(inplace=True),

            nn.Conv2d(512, 256, 3, padding=2, dilation=2),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 128, 3, padding=2, dilation=2),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 64, 3, padding=2, dilation=2),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 1, 1)
        )

        # ✅ Weight initialization for backend
        self._initialize_backend_weights()

    def forward(self, x):
        x = self.frontend(x)
        x = self.backend(x)
        return x

    def _initialize_backend_weights(self):
        for m in self.backend.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, std=0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

In [7]:
#Model Training setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Pointing to the specific subfolders
BASE = "/kaggle/input/datasets/vikasn2006/shanghai-crowd"

img_dir = f"{BASE}/ShanghaiTech/part_B/train_data/images"
gt_dir = f"{BASE}/ShanghaiTech/part_B/train_data/ground-truth"

train_dataset = CrowdDataset(img_dir, mat_folder=gt_dir)

if len(train_dataset) == 0:
    raise ValueError(f"No images found in {img_dir}. Check your paths.")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)

model = CSRNet().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = nn.DataParallel(model)

Using 2 GPUs


In [8]:
epochs = 35
import torch
torch.cuda.empty_cache()

for epoch in range(epochs):
    model.train()
    epoch_loss = 0

    for images, targets in tqdm(train_loader):
        images = images.to(device)
        targets = targets.to(device)

        outputs = model(images)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {epoch_loss}")

 28%|██▊       | 14/50 [00:21<00:55,  1.54s/it]


KeyboardInterrupt: 

In [ ]:
model.eval()
mae = 0
mse = 0
total_samples = 0

with torch.no_grad():
    for images, targets in train_loader:
        images = images.to(device)
        targets = targets.to(device)

        outputs = model(images)

        for i in range(outputs.shape[0]):
            pred_count = outputs[i].sum().item()
            gt_count = targets[i].sum().item()

            mae += abs(pred_count - gt_count)
            mse += (pred_count - gt_count) ** 2
            total_samples += 1

mae = mae / total_samples
rmse = np.sqrt(mse / total_samples)

print("MAE:", mae)
print("RMSE:", rmse)

In [ ]:
torch.save(model.state_dict(), "/kaggle/working/csrnet_model.pth")

In [ ]:
results = []

# start time for synthetic timestamps
timestamps = pd.date_range(
    start="2025-01-01 08:00:00",
    periods=len(train_dataset),
    freq="1min"
)

counter = 0

model.eval()
with torch.no_grad():
    for batch_idx, (images, targets) in enumerate(train_loader):

        images = images.to(device)
        targets = targets.to(device)

        outputs = model(images)

        for i in range(outputs.shape[0]):

            predicted_count = outputs[i].sum().item()
            actual_count = targets[i].sum().item()

            results.append({
                "timestamp": timestamps[counter],
                "image_id": counter,
                "predicted_count": predicted_count,
                "actual_count": actual_count
            })

            counter += 1

df = pd.DataFrame(results)

df.to_csv("/kaggle/working/crowd_counts.csv", index=False)

print("CSV exported successfully")

import matplotlib.pyplot as plt

plt.scatter(df["actual_count"], df["predicted_count"], alpha=0.6)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.plot([0, max(df["actual_count"])],
         [0, max(df["actual_count"])],
         'r')
plt.show()

errors = np.array(predicted_count) - np.array(actual_count)

plt.hist(errors, bins=30)
plt.title("Error Distribution")
plt.xlabel("Prediction Error")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Pointing to the specific subfolders
BASE = "/kaggle/input/datasets/vikasn2006/shanghai-crowd"

test_img_dir = f"{BASE}/ShanghaiTech/part_B/test_data/images"
test_gt_dir = f"{BASE}/ShanghaiTech/part_B/test_data/ground-truth"

test_dataset = CrowdDataset(test_img_dir, mat_folder=test_gt_dir)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
# ========== 1.3 Comprehensive CSRNet Evaluation ==========
def calculate_metrics(predictions, targets):
    """Calculate multiple evaluation metrics"""
    mae = np.mean(np.abs(predictions - targets))
    rmse = np.sqrt(np.mean((predictions - targets) ** 2))
    mape = np.mean(np.abs((targets - predictions) / (targets + 1e-8))) * 100
    r2 = 1 - np.sum((targets - predictions) ** 2) / np.sum((targets - np.mean(targets)) ** 2)
    
    return mae, rmse, mape, r2

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device)
        outputs = model(images)
        
        for i in range(outputs.shape[0]):
            pred_count = outputs[i].sum().item()
            gt_count = targets[i].sum().item()
            all_preds.append(pred_count)
            all_targets.append(gt_count)

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

mae, rmse, mape, r2 = calculate_metrics(all_preds, all_targets)

print("\n========== CSRNet Test Results ==========")
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")
print(f"R²:   {r2:.4f}")
print("=" * 40)

# Save results
with open('/kaggle/working/csrnet_results.txt', 'w') as f:
    f.write(f"CSRNet Test Results\n")
    f.write(f"MAE: {mae:.2f}\n")
    f.write(f"RMSE: {rmse:.2f}\n")
    f.write(f"MAPE: {mape:.2f}%\n")
    f.write(f"R²: {r2:.4f}\n")

In [ ]:
image, gt_density = train_dataset[0]

model.eval()
with torch.no_grad():
    pred_density = model(image.unsqueeze(0).to(device))

pred_density = pred_density.squeeze().cpu().numpy()
gt_density = gt_density.squeeze().numpy()
 
plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.title("Original Image")
plt.imshow(image.permute(1,2,0))

plt.subplot(1,3,2)
plt.title("Ground Truth Density")
plt.imshow(gt_density, cmap="jet")

plt.subplot(1,3,3)
plt.title("Predicted Density")
plt.imshow(pred_density, cmap="jet")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

model.eval()

with torch.no_grad():
    image, gt_density = train_dataset[0]
    image_input = image.unsqueeze(0).to(device)

    pred_density = model(image_input)
    pred_density = pred_density.squeeze().cpu().numpy()
    gt_density = gt_density.squeeze().numpy()

    # Original Image
    plt.figure(figsize=(15,5))

    plt.subplot(1,3,1)
    plt.title("Original Image")
    plt.imshow(image.permute(1,2,0))
    plt.axis('off')

    # Ground Truth Density
    plt.subplot(1,3,2)
    plt.title("Ground Truth Density")
    plt.imshow(gt_density, cmap='jet')
    plt.colorbar()
    plt.axis('off')

    # Predicted Density
    plt.subplot(1,3,3)
    plt.title("Predicted Density")
    plt.imshow(pred_density, cmap='jet')
    plt.colorbar()
    plt.axis('off')

    plt.show()

In [ ]:
loss_history = []

for epoch in range(epochs):
    ...
    loss_history.append(epoch_loss)

plt.plot(loss_history)
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
df = pd.read_csv("/kaggle/working/temple_footfall_realtime.csv")

# Use predicted counts for modeling
counts = df["footfall"].values

print(counts[:10])

In [ ]:
# ========== 3.1 ARIMA Model ==========
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

# Prepare data (use original count series)
train_counts = counts[:train_size].flatten()
test_counts = counts[train_size:train_size+test_size].flatten()

# Check stationarity
result = adfuller(train_counts)
print(f'ADF Statistic: {result[0]:.4f}')
print(f'p-value: {result[1]:.4f}')

# If not stationary, differencing needed
# Find best ARIMA order using AIC
best_aic = float('inf')
best_order = None

# Try different orders (simplified for time)
p_values = range(0, 3)
d_values = range(0, 2)
q_values = range(0, 3)

for p in p_values:
    for d in d_values:
        for q in q_values:
            try:
                model = ARIMA(train_counts, order=(p, d, q))
                model_fit = model.fit()
                if model_fit.aic < best_aic:
                    best_aic = model_fit.aic
                    best_order = (p, d, q)
            except:
                continue

print(f"Best ARIMA order: {best_order}, AIC: {best_aic:.2f}")

# Fit final ARIMA model
final_arima = ARIMA(train_counts, order=best_order)
arima_fit = final_arima.fit()

# Forecast
forecast = arima_fit.forecast(steps=len(test_counts))

# Calculate metrics
mae_arima = np.mean(np.abs(forecast - test_counts))
rmse_arima = np.sqrt(np.mean((forecast - test_counts) ** 2))
mape_arima = np.mean(np.abs((test_counts - forecast) / (test_counts + 1e-8))) * 100
r2_arima = 1 - np.sum((test_counts - forecast) ** 2) / np.sum((test_counts - np.mean(test_counts)) ** 2)

print("\n========== ARIMA Results ==========")
print(f"MAE:  {mae_arima:.2f}")
print(f"RMSE: {rmse_arima:.2f}")
print(f"MAPE: {mape_arima:.2f}%")
print(f"R²:   {r2_arima:.4f}")
print("=" * 40)

In [ ]:
# ========== 3.2 GRU Model (Evaluation) ==========
# Load best GRU model
gru_model.load_state_dict(torch.load('/kaggle/working/best_gru.pth'))
gru_model.eval()

# Prepare lists to collect predictions and actuals
all_gru_preds = []
all_actuals = []

with torch.no_grad():
    for batch_X, batch_y in test_loader_final:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        outputs = gru_model(batch_X)
        all_gru_preds.append(outputs.cpu().numpy())
        all_actuals.append(batch_y.cpu().numpy())

# Concatenate batch predictions and actuals
gru_predictions_scaled = np.concatenate(all_gru_preds, axis=0)
actual_scaled = np.concatenate(all_actuals, axis=0)

# Inverse transform to original scale
gru_predictions = scaler.inverse_transform(gru_predictions_scaled)
actual = scaler.inverse_transform(actual_scaled)

# Calculate metrics
mae_gru = np.mean(np.abs(gru_predictions - actual))
rmse_gru = np.sqrt(np.mean((gru_predictions - actual) ** 2))
mape_gru = np.mean(np.abs((actual - gru_predictions) / (actual + 1e-8))) * 100
r2_gru = 1 - np.sum((actual - gru_predictions) ** 2) / np.sum((actual - np.mean(actual)) ** 2)

print("\n========== GRU Test Results ==========")
print(f"MAE:  {mae_gru:.2f}")
print(f"RMSE: {rmse_gru:.2f}")
print(f"MAPE: {mape_gru:.2f}%")
print(f"R²:   {r2_gru:.4f}")
print("=" * 40)

In [ ]:
# ========== 3.3 Linear Regression Baseline ==========
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# Prepare lagged features (use last 5 values as features)
def create_lag_features(data, n_lags=5):
    X_lag = []
    y_lag = []
    for i in range(len(data) - n_lags):
        X_lag.append(data[i:i+n_lags])
        y_lag.append(data[i+n_lags])
    return np.array(X_lag), np.array(y_lag)

# Use scaled data
X_lr, y_lr = create_lag_features(counts_scaled.flatten(), n_lags=5)

# Split
train_size_lr = int(0.7 * len(X_lr))
val_size_lr = int(0.15 * len(X_lr))

X_train_lr = X_lr[:train_size_lr]
X_val_lr = X_lr[train_size_lr:train_size_lr+val_size_lr]
X_test_lr = X_lr[train_size_lr+val_size_lr:]
y_train_lr = y_lr[:train_size_lr]
y_val_lr = y_lr[train_size_lr:train_size_lr+val_size_lr]
y_test_lr = y_lr[train_size_lr+val_size_lr:]

# Train linear regression
lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train_lr)

# Predict and inverse transform
lr_pred_scaled = lr_model.predict(X_test_lr)
lr_pred = scaler.inverse_transform(lr_pred_scaled.reshape(-1, 1))
actual_lr = scaler.inverse_transform(y_test_lr.reshape(-1, 1))

mae_lr = np.mean(np.abs(lr_pred - actual_lr))
rmse_lr = np.sqrt(np.mean((lr_pred - actual_lr) ** 2))
mape_lr = np.mean(np.abs((actual_lr - lr_pred) / (actual_lr + 1e-8))) * 100
r2_lr = 1 - np.sum((actual_lr - lr_pred) ** 2) / np.sum((actual_lr - np.mean(actual_lr)) ** 2)

print("\n========== Linear Regression Results ==========")
print(f"MAE:  {mae_lr:.2f}")
print(f"RMSE: {rmse_lr:.2f}")
print(f"MAPE: {mape_lr:.2f}%")
print(f"R²:   {r2_lr:.4f}")
print("=" * 40)

In [ ]:
# ========== 4.1 Define Risk Levels from Ground Truth ==========
# Define risk thresholds based on actual counts
def get_risk_level(count):
    if count < 100:
        return 0  # Safe
    elif count < 200:
        return 1  # Warning
    else:
        return 2  # Critical

# Create ground truth risk labels (from actual counts)
actual_risk_labels = [get_risk_level(val[0]) for val in actual]

# Create predicted risk scores (from your heuristic formula)
# Or use predictions directly as scores
risk_scores = predictions.flatten()  # Use predicted counts as risk scores

print(f"Risk distribution: Safe: {actual_risk_labels.count(0)}, "
      f"Warning: {actual_risk_labels.count(1)}, "
      f"Critical: {actual_risk_labels.count(2)}")

In [ ]:
# ========== 5.1 Training Loss Curves ==========
plt.figure(figsize=(15, 5))

# CSRNet Loss
plt.subplot(1, 3, 1)
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CSRNet Training Curves')
plt.legend()
plt.grid(True, alpha=0.3)

# LSTM Loss
plt.subplot(1, 3, 2)
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('LSTM Training Curves')
plt.legend()
plt.grid(True, alpha=0.3)

# GRU Loss
plt.subplot(1, 3, 3)
plt.plot(gru_train_losses, label='Train Loss', linewidth=2)
plt.plot(gru_val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('GRU Training Curves')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ========== 5.2 Time Series Predictions Comparison ==========
plt.figure(figsize=(15, 6))

# Plot all predictions
plt.plot(actual, label='Actual', linewidth=2, alpha=0.7)
plt.plot(predictions, label='LSTM Predictions', linewidth=1.5, alpha=0.8)
plt.plot(gru_predictions, label='GRU Predictions', linewidth=1.5, alpha=0.8)
plt.plot(lr_pred, label='Linear Regression', linewidth=1.5, alpha=0.8)

plt.xlabel('Time Steps (Test Set)')
plt.ylabel('Crowd Count')
plt.title('Time Series Predictions - Model Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

# Add shaded risk zones
plt.axhspan(0, 100, alpha=0.2, color='green', label='Safe Zone')
plt.axhspan(100, 200, alpha=0.2, color='yellow', label='Warning Zone')
plt.axhspan(200, actual.max(), alpha=0.2, color='red', label='Critical Zone')

plt.tight_layout()
plt.savefig('/kaggle/working/time_series_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ========== 5.3 Bar Chart Comparison ==========
models = ['LSTM', 'GRU', 'ARIMA', 'Linear Regression']
mae_values = [mae_lstm, mae_gru, mae_arima, mae_lr]
rmse_values = [rmse_lstm, rmse_gru, rmse_arima, rmse_lr]

x = np.arange(len(models))
width = 0.


In [ ]:
import matplotlib.pyplot as plt

plt.plot(actual, label="Actual")
plt.plot(predictions, label="Predicted")
plt.legend()
plt.title("Crowd Prediction (LSTM)")
plt.show()

In [ ]:
def compute_risk(current, growth, predicted):

    risk = 0.4*current + 0.3*growth + 0.3*predicted + np.random.normal(0, 5)

    return risk

In [ ]:
def classify_risk(score):
    if score < 100:
        return "Safe"
    elif score < 200:
        return "Warning"
    else:
        return "Critical"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import random

# ============================================
# SIMULATED REALTIME TEMPLE FOOTFALL DATA
# ============================================

class TempleFootfallSimulator:
    """
    Simulates real-time footfall data for a temple with realistic patterns:
    - Diurnal pattern (morning and evening peaks)
    - Weekly pattern (weekends busier)
    - Festival effects
    - Random variations
    """
    
    def __init__(self, start_date="2024-01-01", days=180):
        self.start_date = pd.to_datetime(start_date)
        self.days = days
        self.current_idx = 0
        
        # Generate base data
        self.df = self._generate_footfall_data()
        
    def _get_hour_factor(self, hour):
        """Returns hourly footfall multiplier based on temple visiting patterns"""
        # Temple patterns: early morning (6-9 AM), evening (5-8 PM)
        # Low during midday and late night
        
        # Morning peak (6-9 AM)
        if 6 <= hour <= 9:
            factor = 1.5 - abs(hour - 7.5) * 0.3
        # Evening peak (5-8 PM)
        elif 17 <= hour <= 20:
            factor = 1.4 - abs(hour - 18.5) * 0.25
        # Morning pre-peak (4-6 AM)
        elif 4 <= hour < 6:
            factor = 0.6
        # Night (9 PM - 4 AM)
        elif hour >= 21 or hour < 4:
            factor = 0.2
        # Midday (10 AM - 4 PM)
        else:
            factor = 0.8
        return factor
    
    def _get_day_factor(self, date):
        """Returns day-of-week multiplier"""
        day = date.dayofweek
        # 0=Monday, 6=Sunday
        if day >= 5:  # Weekend
            return 1.4
        elif day == 4:  # Friday (special in many temples)
            return 1.2
        else:
            return 1.0
    
    def _get_festival_factor(self, date):
        """Simulates festival effects on specific dates"""
        festivals = {
            # Major festivals (2-3x normal)
            "2024-01-15": 2.5,   # Pongal
            "2024-03-25": 2.8,   # Holi
            "2024-04-09": 2.0,   # Ugadi
            "2024-08-15": 2.2,   # Independence Day
            "2024-09-07": 3.0,   # Ganesh Chaturthi
            "2024-10-12": 2.5,   # Dussehra
            "2024-11-01": 2.0,   # Diwali start
            "2024-11-02": 3.0,   # Diwali peak
            "2024-12-25": 2.0,   # Christmas
            
            # Weekend festivals (even higher)
            "2024-01-20": 1.8,   # Local festival
            "2024-02-14": 1.5,   # Valentine's Day
            "2024-03-01": 1.8,   # Maha Shivratri
            "2024-04-14": 2.0,   # Tamil New Year
            "2024-06-21": 1.5,   # Yoga Day
            "2024-08-26": 1.8,   # Janmashtami
        }
        
        date_str = date.strftime("%Y-%m-%d")
        
        # Check for exact date match
        if date_str in festivals:
            return festivals[date_str]
        
        # Weekend festivals get slight boost
        if date.dayofweek >= 5:
            return 1.1
        
        return 1.0
    
    def _get_weather_factor(self, date):
        """Simulates weather impact (simplified seasonal pattern)"""
        month = date.month
        
        # Monsoon season (June-September) - lower footfall on rainy days
        if 6 <= month <= 9:
            # Simulate random rainy days (30% chance of rain)
            rainy = random.random() < 0.3
            return 0.7 if rainy else 0.95
        
        # Summer (April-May) - lower during peak heat
        elif month in [4, 5]:
            return 0.9
        
        # Winter (December-February) - slightly higher
        elif month in [12, 1, 2]:
            return 1.05
        
        return 1.0
    
    def _generate_footfall_data(self):
        """Generate complete footfall dataset"""
        timestamps = []
        footfall_values = []
        
        # Base footfall (people per hour)
        base_footfall = 100
        
        for day in range(self.days):
            current_date = self.start_date + timedelta(days=day)
            
            # Get daily factors
            day_factor = self._get_day_factor(current_date)
            festival_factor = self._get_festival_factor(current_date)
            weather_factor = self._get_weather_factor(current_date)
            
            for hour in range(24):
                timestamp = current_date + timedelta(hours=hour)
                hour_factor = self._get_hour_factor(hour)
                
                # Calculate footfall
                footfall = base_footfall
                footfall *= hour_factor
                footfall *= day_factor
                footfall *= festival_factor
                footfall *= weather_factor
                
                # Add random variation (±15%)
                variation = 1 + random.uniform(-0.15, 0.15)
                footfall *= variation
                
                # Add occasional spike for special events (random ceremonies)
                if random.random() < 0.02:  # 2% chance per hour
                    spike = random.uniform(1.3, 2.0)
                    footfall *= spike
                
                footfall = max(0, int(footfall))
                
                timestamps.append(timestamp)
                footfall_values.append(footfall)
        
        return pd.DataFrame({
            'timestamp': timestamps,
            'footfall': footfall_values,
            'hour': [ts.hour for ts in timestamps],
            'day_of_week': [ts.dayofweek for ts in timestamps],
            'month': [ts.month for ts in timestamps]
        })
    
    def get_next_hour(self):
        """Simulates real-time data streaming (one hour at a time)"""
        if self.current_idx >= len(self.df):
            return None
        
        row = self.df.iloc[self.current_idx]
        self.current_idx += 1
        
        return {
            'timestamp': row['timestamp'],
            'footfall': row['footfall'],
            'hour': row['hour'],
            'day_of_week': row['day_of_week'],
            'month': row['month']
        }
    
    def get_recent_data(self, hours=24):
        """Get recent data for model input"""
        start_idx = max(0, self.current_idx - hours)
        return self.df.iloc[start_idx:self.current_idx]


# ============================================
# CREATE SIMULATED DATASET
# ============================================

# Initialize simulator
simulator = TempleFootfallSimulator(start_date="2024-01-01", days=180)

# Get full dataset
df_realtime = simulator.df.copy()

print("=" * 60)
print("SIMULATED REALTIME TEMPLE FOOTFALL DATA")
print("=" * 60)
print(f"Date Range: {df_realtime['timestamp'].min()} to {df_realtime['timestamp'].max()}")
print(f"Total Records: {len(df_realtime):,}")
print(f"Average Footfall: {df_realtime['footfall'].mean():.0f} people/hour")
print(f"Max Footfall: {df_realtime['footfall'].max():.0f}")
print(f"Min Footfall: {df_realtime['footfall'].min():.0f}")
print("=" * 60)


# ============================================
# VISUALIZE PATTERNS
# ============================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Hourly Pattern (Typical Day)
ax1 = axes[0, 0]
hourly_avg = df_realtime.groupby('hour')['footfall'].mean()
ax1.plot(hourly_avg.index, hourly_avg.values, 'b-o', linewidth=2, markersize=4)
ax1.axvspan(5, 9, alpha=0.3, color='yellow', label='Morning Peak')
ax1.axvspan(16, 20, alpha=0.3, color='orange', label='Evening Peak')
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Average Footfall (people/hour)')
ax1.set_title('Typical Daily Pattern - Temple Footfall')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Weekly Pattern
ax2 = axes[0, 1]
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily_avg = df_realtime.groupby('day_of_week')['footfall'].mean()
ax2.bar(day_names, daily_avg.values, color='skyblue', edgecolor='navy')
ax2.set_xlabel('Day of Week')
ax2.set_ylabel('Average Footfall (people/hour)')
ax2.set_title('Weekly Pattern - Weekend Surge')
ax2.grid(True, alpha=0.3, axis='y')

# 3. Time Series (First 30 days)
ax3 = axes[1, 0]
first_30_days = df_realtime[df_realtime['timestamp'] < df_realtime['timestamp'].min() + timedelta(days=30)]
ax3.plot(first_30_days['timestamp'], first_30_days['footfall'], 'g-', linewidth=1, alpha=0.7)
ax3.set_xlabel('Date')
ax3.set_ylabel('Footfall (people/hour)')
ax3.set_title('Footfall Time Series (First 30 Days)')
ax3.tick_params(axis='x', rotation=45)
ax3.grid(True, alpha=0.3)

# 4. Risk Level Distribution
ax4 = axes[1, 1]
safe = df_realtime[df_realtime['footfall'] < 100].shape[0]
warning = df_realtime[(df_realtime['footfall'] >= 100) & (df_realtime['footfall'] < 200)].shape[0]
critical = df_realtime[df_realtime['footfall'] >= 200].shape[0]

colors = ['green', 'orange', 'red']
labels = [f'Safe (<100): {safe/len(df_realtime)*100:.1f}%',
          f'Warning (100-200): {warning/len(df_realtime)*100:.1f}%',
          f'Critical (>200): {critical/len(df_realtime)*100:.1f}%']

ax4.pie([safe, warning, critical], labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax4.set_title('Risk Level Distribution')

plt.tight_layout()
plt.show()


# ============================================
# DEMONSTRATE REALTIME STREAMING
# ============================================

print("\n" + "=" * 60)
print("REALTIME DATA STREAM SIMULATION")
print("=" * 60)

# Reset simulator
simulator = TempleFootfallSimulator(start_date="2024-10-01", days=7)

print("\nSimulating 48 hours of realtime data...\n")

# Collect predictions for comparison
timestamps = []
actual_values = []
predictions = []

for i in range(48):
    data = simulator.get_next_hour()
    if data is None:
        break
    
    timestamps.append(data['timestamp'])
    actual_values.append(data['footfall'])
    
    # Simple prediction: weighted average of last 24 hours
    if len(actual_values) > 24:
        recent = actual_values[-24:]
        weights = np.exp(np.linspace(-1, 0, 24))
        weights = weights / weights.sum()
        pred = np.sum(np.array(recent) * weights)
        predictions.append(pred)
    else:
        predictions.append(actual_values[-1])
    
    # Print every 6 hours
    if i % 6 == 0:
        print(f"{data['timestamp']} | Footfall: {data['footfall']:4d} | "
              f"Risk: {'SAFE' if data['footfall'] < 100 else 'WARNING' if data['footfall'] < 200 else 'CRITICAL'}")


# ============================================
# VISUALIZE REALTIME PREDICTIONS
# ============================================

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(timestamps, actual_values, 'b-', linewidth=2, label='Actual Footfall', alpha=0.8)
ax.plot(timestamps, predictions, 'r--', linewidth=1.5, label='Predicted (EMA)', alpha=0.8)

# Add risk zones
ax.axhspan(0, 100, alpha=0.2, color='green', label='Safe Zone')
ax.axhspan(100, 200, alpha=0.2, color='yellow', label='Warning Zone')
ax.axhspan(200, max(actual_values + predictions), alpha=0.2, color='red', label='Critical Zone')

ax.set_xlabel('Time')
ax.set_ylabel('Footfall (people/hour)')
ax.set_title('Realtime Temple Footfall - 48 Hours')
ax.legend(loc='upper left')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# ============================================
# EXPORT DATA FOR LSTM TRAINING
# ============================================

# Save full dataset for model training
df_realtime.to_csv('/kaggle/working/temple_footfall_realtime.csv', index=False)
print(f"\n✅ Data saved to: /kaggle/working/temple_footfall_realtime.csv")

# Create a smaller recent dataset for testing
recent_data = simulator.get_recent_data(hours=168)  # Last 7 days
recent_data.to_csv('/kaggle/working/temple_footfall_recent.csv', index=False)
print(f"✅ Recent data (7 days) saved to: /kaggle/working/temple_footfall_recent.csv")


# ============================================
# SUMMARY STATISTICS
# ============================================

print("\n" + "=" * 60)
print("DATA SUMMARY")
print("=" * 60)

# Peak hours analysis
peak_hours = df_realtime.groupby('hour')['footfall'].mean().nlargest(5)
print("\nTop 5 Peak Hours:")
for hour, count in peak_hours.items():
    print(f"  {hour:02d}:00 - {count:.0f} people/hour")

# Busiest days
busiest_days = df_realtime.groupby(df_realtime['timestamp'].dt.date)['footfall'].sum().nlargest(5)
print("\nBusiest Days:")
for date, total in busiest_days.items():
    print(f"  {date}: {total:,.0f} total visitors")

# Risk hours
high_risk_hours = df_realtime[df_realtime['footfall'] >= 200].groupby('hour').size().sort_values(ascending=False)
print("\nHigh Risk Hours (>200 people/hour):")
for hour, count in high_risk_hours.head(5).items():
    print(f"  {hour:02d}:00 - {count} occurrences")

print("\n" + "=" * 60)

In [ ]:
# ============================================
# COMPLETE LSTM IMPLEMENTATION - CORRECTED VERSION
# ============================================

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# ============================================
# PART 1: GENERATE SIMULATED TEMPLE FOOTFALL DATA
# ============================================

class TempleFootfallSimulator:
    """Generates realistic temple footfall data with patterns"""
    
    def __init__(self, start_date="2024-01-01", days=365):
        self.start_date = pd.to_datetime(start_date)
        self.days = days
        
    def generate_data(self):
        """Generate complete footfall dataset"""
        timestamps = []
        footfall_values = []
        
        base_footfall = 100  # Base people per hour
        
        for day in range(self.days):
            current_date = self.start_date + timedelta(days=day)
            
            # Day factor (weekends busier)
            day_factor = 1.4 if current_date.dayofweek >= 5 else 1.0
            
            # Friday special
            if current_date.dayofweek == 4:
                day_factor *= 1.2
            
            # Festival effects
            festivals = {
                "2024-01-15": 2.5, "2024-03-25": 2.8, "2024-08-15": 2.2,
                "2024-09-07": 3.0, "2024-10-12": 2.5, "2024-11-02": 3.0,
                "2024-12-25": 2.0
            }
            festival_factor = festivals.get(current_date.strftime("%Y-%m-%d"), 1.0)
            
            for hour in range(24):
                timestamp = current_date + timedelta(hours=hour)
                
                # Hour factor (morning and evening peaks)
                if 6 <= hour <= 9:
                    hour_factor = 1.5 - abs(hour - 7.5) * 0.3
                elif 17 <= hour <= 20:
                    hour_factor = 1.4 - abs(hour - 18.5) * 0.25
                elif hour >= 21 or hour < 4:
                    hour_factor = 0.2
                elif 4 <= hour < 6:
                    hour_factor = 0.6
                else:
                    hour_factor = 0.8
                
                # Calculate footfall
                footfall = base_footfall * hour_factor * day_factor * festival_factor
                
                # Add random variation
                footfall *= 1 + np.random.uniform(-0.15, 0.15)
                
                # Add occasional spikes
                if np.random.random() < 0.02:
                    footfall *= np.random.uniform(1.3, 2.0)
                
                timestamps.append(timestamp)
                footfall_values.append(max(0, int(footfall)))
        
        return pd.DataFrame({
            'timestamp': timestamps,
            'footfall': footfall_values,
            'hour': [ts.hour for ts in timestamps],
            'day_of_week': [ts.dayofweek for ts in timestamps],
            'month': [ts.month for ts in timestamps],
            'is_weekend': [1 if ts.dayofweek >= 5 else 0 for ts in timestamps]
        })

# Generate data
print("=" * 60)
print("GENERATING SIMULATED TEMPLE FOOTFALL DATA")
print("=" * 60)

simulator = TempleFootfallSimulator(start_date="2024-01-01", days=180)
df = simulator.generate_data()

print(f"Data shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Total records: {len(df):,}")
print(f"Average footfall: {df['footfall'].mean():.0f} people/hour")
print(f"Max footfall: {df['footfall'].max():.0f}")
print(f"Min footfall: {df['footfall'].min():.0f}")

# ============================================
# PART 2: DATA PREPROCESSING FOR LSTM
# ============================================

# Add lag features (past values)
def create_lag_features(df, target_col='footfall', lags=[1, 2, 3, 6, 12, 24]):
    """Create lag features for time series prediction"""
    df = df.copy()
    for lag in lags:
        df[f'lag_{lag}'] = df[target_col].shift(lag)
    return df

# Add rolling statistics
def add_rolling_features(df, target_col='footfall', windows=[3, 6, 12, 24]):
    """Add rolling mean and std features"""
    df = df.copy()
    for window in windows:
        df[f'rolling_mean_{window}'] = df[target_col].rolling(window=window).mean()
        df[f'rolling_std_{window}'] = df[target_col].rolling(window=window).std()
    return df

# Add time features
def add_time_features(df):
    """Add cyclical time features for better pattern learning"""
    df = df.copy()
    # Hour (sin/cos for cyclical encoding)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    
    # Day of week (sin/cos for cyclical encoding)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    
    # Month (sin/cos for seasonal patterns)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    
    return df

# Apply feature engineering
print("\n" + "=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

df_features = create_lag_features(df, lags=[1, 2, 3, 6, 12, 24])
df_features = add_rolling_features(df_features, windows=[3, 6, 12, 24])
df_features = add_time_features(df_features)

# Drop NaN values from lag/rolling features
df_features = df_features.dropna().reset_index(drop=True)

print(f"Features created: {len([col for col in df_features.columns if col not in ['timestamp', 'footfall']])}")
print(f"Final data shape: {df_features.shape}")

# ============================================
# PART 3: PREPARE DATA FOR LSTM
# ============================================

# Define features to use
feature_cols = [
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_24',
    'rolling_mean_3', 'rolling_mean_6', 'rolling_mean_12', 'rolling_mean_24',
    'rolling_std_3', 'rolling_std_6', 'rolling_std_12', 'rolling_std_24'
]

target_col = 'footfall'

# Scale features
feature_scaler = StandardScaler()
target_scaler = MinMaxScaler()

X_scaled = feature_scaler.fit_transform(df_features[feature_cols])
y_scaled = target_scaler.fit_transform(df_features[[target_col]])

# Create sequences
def create_sequences(X, y, seq_length=24):
    """Create sequences for LSTM training"""
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length])
    return np.array(X_seq), np.array(y_seq)

SEQ_LENGTH = 24  # Use past 24 hours to predict next hour

X_seq, y_seq = create_sequences(X_scaled, y_scaled, SEQ_LENGTH)

print(f"\nSequence shape: {X_seq.shape}")
print(f"Target shape: {y_seq.shape}")

# Train/validation/test split
train_size = int(0.7 * len(X_seq))
val_size = int(0.15 * len(X_seq))
test_size = len(X_seq) - train_size - val_size

X_train = X_seq[:train_size]
y_train = y_seq[:train_size]
X_val = X_seq[train_size:train_size+val_size]
y_val = y_seq[train_size:train_size+val_size]
X_test = X_seq[train_size+val_size:]
y_test = y_seq[train_size+val_size:]

print(f"\nTrain size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Create data loaders
batch_size = 32
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ============================================
# PART 4: ENHANCED LSTM MODEL ARCHITECTURE
# ============================================

class EnhancedCrowdLSTM(nn.Module):
    """
    Enhanced LSTM for crowd footfall prediction with:
    - Bidirectional LSTM layers
    - Attention mechanism
    - Residual connections
    - Dropout for regularization
    """
    
    def __init__(self, input_size, hidden_size=128, num_layers=3, dropout=0.3, bidirectional=True):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        
        # Attention mechanism
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * self.num_directions, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_size * self.num_directions, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 1)
        
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.batch_norm1 = nn.BatchNorm1d(128)
        self.batch_norm2 = nn.BatchNorm1d(64)
        self.batch_norm3 = nn.BatchNorm1d(32)
        
    def forward(self, x):
        # LSTM layer
        lstm_out, (hidden, cell) = self.lstm(x)
        
        # Attention mechanism
        attention_weights = self.attention(lstm_out)
        attention_weights = torch.softmax(attention_weights, dim=1)
        context = torch.sum(attention_weights * lstm_out, dim=1)
        
        # Fully connected layers with batch norm and dropout
        out = self.fc1(context)
        out = self.batch_norm1(out)
        out = self.relu(out)
        out = self.dropout(out)
        
        out = self.fc2(out)
        out = self.batch_norm2(out)
        out = self.relu(out)
        out = self.dropout(out)
        
        out = self.fc3(out)
        out = self.batch_norm3(out)
        out = self.relu(out)
        out = self.dropout(out)
        
        out = self.fc4(out)
        
        return out


# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_size = len(feature_cols)

model = EnhancedCrowdLSTM(
    input_size=input_size,
    hidden_size=128,
    num_layers=3,
    dropout=0.3,
    bidirectional=True
).to(device)

print("\n" + "=" * 60)
print("MODEL ARCHITECTURE")
print("=" * 60)
print(f"Device: {device}")
print(f"Input size: {input_size}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# ============================================
# PART 5: TRAINING SETUP (FIXED)
# ============================================

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# Fixed: Remove verbose parameter (deprecated in newer PyTorch)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=10, factor=0.5
)

# Early stopping
class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.0001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model_state = None
        
    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict().copy()
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict().copy()
            self.counter = 0
    
    def get_best_model(self):
        return self.best_model_state

early_stopping = EarlyStopping(patience=15)

# ============================================
# PART 6: TRAINING LOOP
# ============================================

print("\n" + "=" * 60)
print("TRAINING LSTM MODEL")
print("=" * 60)

epochs = 100
train_losses = []
val_losses = []

for epoch in range(epochs):
    # Training phase
    model.train()
    train_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            output = model(batch_X)
            val_loss += criterion(output, batch_y).item()
    
    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    # Learning rate scheduling
    scheduler.step(avg_val_loss)
    
    # Early stopping
    early_stopping(avg_val_loss, model)
    
    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")
    
    if early_stopping.early_stop:
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        break

# Load best model
model.load_state_dict(early_stopping.get_best_model())

print(f"\nTraining completed! Best validation loss: {early_stopping.best_loss:.6f}")

# ============================================
# PART 7: MODEL EVALUATION
# ============================================

def evaluate_model(model, test_loader, target_scaler):
    """Evaluate model on test data"""
    model.eval()
    predictions = []
    actuals = []
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            output = model(batch_X)
            predictions.extend(output.cpu().numpy())
            actuals.extend(batch_y.numpy())
    
    # Inverse transform
    predictions = target_scaler.inverse_transform(np.array(predictions).reshape(-1, 1))
    actuals = target_scaler.inverse_transform(np.array(actuals).reshape(-1, 1))
    
    # Calculate metrics
    mae = mean_absolute_error(actuals, predictions)
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    r2 = r2_score(actuals, predictions)
    
    # MAPE (avoid division by zero)
    mape = np.mean(np.abs((actuals - predictions) / (actuals + 1))) * 100
    
    return predictions, actuals, {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}

# Evaluate
predictions, actuals, metrics = evaluate_model(model, test_loader, target_scaler)

print("\n" + "=" * 60)
print("TEST RESULTS")
print("=" * 60)
print(f"MAE:  {metrics['MAE']:.2f} people/hour")
print(f"RMSE: {metrics['RMSE']:.2f} people/hour")
print(f"MAPE: {metrics['MAPE']:.2f}%")
print(f"R²:   {metrics['R2']:.4f}")

# ============================================
# PART 8: VISUALIZATIONS
# ============================================

# 8.1 Training Curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Training loss
ax1 = axes[0, 0]
ax1.plot(train_losses, label='Train Loss', linewidth=2)
ax1.plot(val_losses, label='Validation Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Predictions vs Actual
ax2 = axes[0, 1]
test_len = min(168, len(actuals))  # Show up to 7 days
ax2.plot(actuals[:test_len], label='Actual', linewidth=1.5, alpha=0.7)
ax2.plot(predictions[:test_len], label='Predicted', linewidth=1.5, alpha=0.7)
ax2.set_xlabel('Time Step (hours)')
ax2.set_ylabel('Footfall (people/hour)')
ax2.set_title('Predictions vs Actual (Test Set)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Scatter plot
ax3 = axes[1, 0]
ax3.scatter(actuals, predictions, alpha=0.5, s=10)
ax3.plot([actuals.min(), actuals.max()], [actuals.min(), actuals.max()], 'r--', linewidth=2, label='Perfect Prediction')
ax3.set_xlabel('Actual Footfall')
ax3.set_ylabel('Predicted Footfall')
ax3.set_title('Prediction Scatter Plot')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Error distribution
ax4 = axes[1, 1]
errors = predictions - actuals
ax4.hist(errors, bins=50, edgecolor='black', alpha=0.7)
ax4.axvline(x=0, color='r', linestyle='--', linewidth=2)
ax4.set_xlabel('Prediction Error (people/hour)')
ax4.set_ylabel('Frequency')
ax4.set_title('Error Distribution')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 8.2 Time Series with Risk Zones
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(actuals[:test_len], 'b-', label='Actual', linewidth=1.5, alpha=0.7)
ax.plot(predictions[:test_len], 'r--', label='Predicted', linewidth=1.5, alpha=0.7)

# Add risk zones
ax.axhspan(0, 100, alpha=0.2, color='green', label='Safe Zone (<100)')
ax.axhspan(100, 200, alpha=0.2, color='yellow', label='Warning Zone (100-200)')
ax.axhspan(200, max(actuals.max(), predictions.max()), alpha=0.2, color='red', label='Critical Zone (>200)')

ax.set_xlabel('Time Step (hours)')
ax.set_ylabel('Footfall (people/hour)')
ax.set_title('LSTM Predictions with Risk Zones')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 8.3 Risk Classification Metrics
def classify_risk(footfall):
    if footfall < 100:
        return 'Safe'
    elif footfall < 200:
        return 'Warning'
    else:
        return 'Critical'

actual_risk = [classify_risk(a[0]) for a in actuals]
pred_risk = [classify_risk(p[0]) for p in predictions]

from sklearn.metrics import confusion_matrix, classification_report

print("\n" + "=" * 60)
print("RISK CLASSIFICATION RESULTS")
print("=" * 60)

print("\nConfusion Matrix:")
print(pd.crosstab(pd.Series(actual_risk, name='Actual'), 
                  pd.Series(pred_risk, name='Predicted')))

print("\nClassification Report:")
print(classification_report(actual_risk, pred_risk))

# ============================================
# PART 9: REALTIME PREDICTION FUNCTION
# ============================================

class RealtimePredictor:
    """Class for making realtime predictions with the trained LSTM"""
    
    def __init__(self, model, feature_scaler, target_scaler, feature_cols, seq_length=24):
        self.model = model
        self.feature_scaler = feature_scaler
        self.target_scaler = target_scaler
        self.feature_cols = feature_cols
        self.seq_length = seq_length
        self.history = []
        
    def update(self, new_data):
        """Update history with new data point"""
        self.history.append(new_data)
        if len(self.history) > self.seq_length * 2:
            self.history.pop(0)
    
    def predict_next(self):
        """Predict next hour's footfall"""
        if len(self.history) < self.seq_length:
            return None
        
        # Get recent data and create features
        recent = self.history[-self.seq_length:]
        recent_df = pd.DataFrame(recent)
        
        # Scale features
        X_scaled = self.feature_scaler.transform(recent_df[self.feature_cols])
        
        # Reshape for LSTM
        X_input = X_scaled.reshape(1, self.seq_length, -1)
        X_tensor = torch.tensor(X_input, dtype=torch.float32).to(device)
        
        # Predict
        self.model.eval()
        with torch.no_grad():
            pred_scaled = self.model(X_tensor).cpu().numpy()
        
        # Inverse transform
        pred = self.target_scaler.inverse_transform(pred_scaled)[0, 0]
        
        return pred
    
    def predict_risk(self, footfall):
        """Classify risk level"""
        if footfall < 100:
            return "SAFE", "green"
        elif footfall < 200:
            return "WARNING", "orange"
        else:
            return "CRITICAL", "red"

# Demo of realtime prediction
print("\n" + "=" * 60)
print("REALTIME PREDICTION DEMO")
print("=" * 60)

# Create a copy of test features for demo
test_start_idx = train_size + val_size
test_features = df_features.iloc[test_start_idx:test_start_idx + len(actuals)].copy()
test_features = test_features.reset_index(drop=True)

# Initialize predictor
predictor = RealtimePredictor(model, feature_scaler, target_scaler, feature_cols, SEQ_LENGTH)

# Simulate realtime data streaming
print("\nSimulating 48 hours of realtime predictions...\n")
print(f"{'Hour':^6} | {'Actual':^8} | {'Predicted':^8} | {'Risk':^10} | {'Match':^6}")
print("-" * 55)

for i in range(min(48, len(test_features))):
    # Get actual data point
    actual = actuals[i][0]
    
    # Update predictor with features
    row_dict = test_features.iloc[i].to_dict()
    predictor.update(row_dict)
    
    # Predict next hour
    if i >= SEQ_LENGTH:
        pred = predictor.predict_next()
        risk, _ = predictor.predict_risk(pred)
        actual_risk, _ = predictor.predict_risk(actual)
        
        match = "✓" if risk == actual_risk else "✗"
        
        print(f"{i:3d}   | {actual:7.0f} | {pred:7.0f} | {risk:^10} | {match:^6}")
    else:
        print(f"{i:3d}   | {actual:7.0f} | {'--':^7} | {'--':^10} | {'warming':^6}")

# ============================================
# PART 10: SAVE MODEL AND RESULTS
# ============================================

# Save model
torch.save(model.state_dict(), '/kaggle/working/lstm_temple_model.pth')
print("\n✅ Model saved to: /kaggle/working/lstm_temple_model.pth")

# Save scalers
import joblib
joblib.dump(feature_scaler, '/kaggle/working/feature_scaler.pkl')
joblib.dump(target_scaler, '/kaggle/working/target_scaler.pkl')
print("✅ Scalers saved to: /kaggle/working/feature_scaler.pkl, target_scaler.pkl")

# Save results
results_df = pd.DataFrame({
    'actual': actuals.flatten(),
    'predicted': predictions.flatten(),
    'error': (predictions - actuals).flatten(),
    'abs_error': np.abs(predictions - actuals).flatten()
})
results_df.to_csv('/kaggle/working/lstm_predictions.csv', index=False)
print("✅ Predictions saved to: /kaggle/working/lstm_predictions.csv")

# Save metrics
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv('/kaggle/working/lstm_metrics.csv', index=False)
print("✅ Metrics saved to: /kaggle/working/lstm_metrics.csv")

print("\n" + "=" * 60)
print("LSTM IMPLEMENTATION COMPLETE!")
print("=" * 60)

In [ ]:
def compute_growth_rate(series):
    growth = []

    for i in range(1, len(series)):
        growth.append(series[i] - series[i-1])

    growth.insert(0, 0)
    return np.array(growth)

growth_rate = compute_growth_rate(counts)
print("Growth rate = ",growth_rate)